# 00 · Setup & provider check
Run this first. It installs libraries, shows which keys are configured, and (in LIVE mode) makes one tiny real call to each provider so key/model problems show up *here* instead of inside the chatbot.

**Pipeline map (notebooks are connected through the shared Chroma store):**
`01 ingest` → `02 retrieve` → `03 augment + generate` → `04 agentic LangGraph` → `05 evaluate`

In [1]:
# Install every library the pipeline needs (safe to re-run)
%pip install -q -r ../requirements.txt

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ---- Shared setup (identical in every notebook) -------------------------------
import os, sys, json, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "sample_data"))

# OFFLINE=True  -> deterministic hash embeddings + mock LLM + local Chroma. No keys, no network.
# OFFLINE=False -> REAL Gemini/Groq/OpenAI + Chroma Cloud from your .env  (set NB_OFFLINE=0 or edit below)
OFFLINE = os.getenv("NB_OFFLINE", "1") == "1"
if OFFLINE:
    os.environ["RAG_OFFLINE"] = "1"
    os.environ["CHROMA_LOCAL_DIR"] = str(ROOT / "chroma_notebook")   # shared by all notebooks
else:
    os.environ["RAG_OFFLINE"] = "0"

from app.config import get_settings
S = get_settings()
print("mode            :", "OFFLINE (mock)" if S.offline else "LIVE providers")
print("vector store    :", "Chroma Cloud" if S.use_chroma_cloud else f"local ({S.chroma_local_dir})")
print("embedding       :", "hash512" if S.offline else S.embedding_provider)
print("LLM providers   :", [p for p, k in (("gemini", S.gemini_key), ("groq", S.groq_key), ("openai", S.openai_key)) if k] or "none (offline mock)")

mode            : OFFLINE (mock)
vector store    : local (/home/claude/proj/v1/rag_service/chroma_notebook)
embedding       : hash512
LLM providers   : ['gemini', 'groq', 'openai']


## Which keys are present? (values are never printed)

In [3]:
for k in ["GEMINI_API_KEY","GROQ_API_KEY","OPENAI_API_KEY","CHROMA_API_KEY","CHROMA_TENANT","CHROMA_DATABASE","INTERNAL_SERVICE_TOKEN"]:
    print(f"{k:24s}", "SET" if os.getenv(k) else "-- missing --")

GEMINI_API_KEY           SET
GROQ_API_KEY             SET
OPENAI_API_KEY           SET
CHROMA_API_KEY           SET
CHROMA_TENANT            SET
CHROMA_DATABASE          SET
INTERNAL_SERVICE_TOKEN   SET


## Embedding model smoke test

In [4]:
from app.providers import get_embeddings, embedding_id
emb = get_embeddings()
v = emb.embed_query("What is photosynthesis?")
print("embedding id:", embedding_id(), "| dimension:", len(v))
assert len(v) > 100

embedding id: hash512 | dimension: 512


## LLM smoke test (light + heavy tier)

In [5]:
from app.providers import get_llm, llm_text
for tier in ("light", "heavy"):
    out = llm_text(get_llm(tier).invoke("Reply with the single word: pong"))
    print(tier, "->", out.strip()[:60])

light -> OK
heavy -> OK


## Vector store connectivity (write → read → delete a probe record)

In [6]:
from app.store import get_store
from langchain_core.documents import Document
st = get_store()
print("mode:", st.mode, "| collection:", st.collection_name, "| heartbeat:", st.heartbeat())
probe = Document(page_content="probe record for connectivity test", metadata={"chunk_id":"__probe__:0","document_id":"__probe__","chunk_index":0,"subject_code":"__PROBE__","is_active":True})
st.upsert([probe]); assert any(d.metadata["document_id"]=="__probe__" for d in st.chunks_of("__probe__"))
st.delete_document("__probe__"); print("probe write/read/delete OK; total chunks now:", st.count())

mode: chroma_local | collection: edu_chunks__hash512 | heartbeat: True
probe write/read/delete OK; total chunks now: 0
